To run this, press "*Runtime*" and press "*Run all*" on a **free** Tesla T4 Google Colab instance!
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

To install Unsloth on your own computer, follow the installation instructions on our Github page [here](https://unsloth.ai/docs/get-started/install).

You will learn how to do [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), & how to save it

### News

Introducing **[Unsloth Desktop](https://unsloth.ai/docs/desktop)**, the first desktop app to run and train models. Free and open-source for macOS, Windows and Linux. [GitHub](https://github.com/unslothai/unsloth) • [Download](https://unsloth.ai/download)

<p>
<a href="https://unsloth.ai/docs/desktop"><img src="https://raw.githubusercontent.com/unslothai/notebooks/refs/heads/main/assets/unsloth-qwen3-8.png" width="350" alt="Introducing Unsloth Desktop"></a>
</p>

Train MoEs - DeepSeek, GLM, Qwen and gpt-oss 12x faster with 35% less VRAM. [Blog](https://unsloth.ai/docs/new/faster-moe)

Ultra Long-Context Reinforcement Learning is here with 7x more context windows! [Blog](https://unsloth.ai/docs/new/grpo-long-context)

New in Reinforcement Learning: [FP8 RL](https://unsloth.ai/docs/new/fp8-reinforcement-learning) • [Vision RL](https://unsloth.ai/docs/new/vision-reinforcement-learning-vlm-rl) • [Standby](https://unsloth.ai/docs/basics/memory-efficient-rl) • [gpt-oss RL](https://unsloth.ai/docs/new/gpt-oss-reinforcement-learning)

Visit our docs for all our [model uploads](https://unsloth.ai/docs/get-started/unsloth-model-catalog) and [notebooks](https://unsloth.ai/docs/get-started/unsloth-notebooks).

### Installation

In [1]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch>=2.8.0" "triton>=3.4.0" {_numpy} {_pil} torchvision bitsandbytes "transformers==4.56.2" \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth" \
        git+https://github.com/triton-lang/triton.git@0add68262ab0a2e33b84524346cb27cbb2787356#subdirectory=python/triton_kernels
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps transformers==4.56.2 "tokenizers>=0.22.0,<=0.23.0" trl==0.22.2 unsloth unsloth_zoo
!uv pip install --no-deps --upgrade "torchao>=0.16.0"

### Unsloth

We're about to demonstrate the power of the new OpenAI GPT-OSS 20B model through an inference example. For our `bnb-4bit` version, use this [notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/GPT_OSS_BNB_(20B)-Inference.ipynb) instead.

**We're using OpenAI's MXFP4 Triton kernels combined with Unsloth's kernels!**

In [2]:
from unsloth import FastLanguageModel
import torch

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/gpt-oss-20b-unsloth-bnb-4bit", # 20B model using bitsandbytes 4bit quantization
    "unsloth/gpt-oss-120b-unsloth-bnb-4bit",
    "unsloth/gpt-oss-20b", # 20B model using MXFP4 format
    "unsloth/gpt-oss-120b",
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gpt-oss-20b",
    dtype = None, # None for auto detection
    max_seq_length = 4096, # Choose any for long context!
    load_in_4bit = False,  # 4 bit quantization to reduce memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    # token = "YOUR_HF_TOKEN", # HF Token for gated models
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.9.12: Fast Gpt_Oss patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gpt_oss won't work! Using float32.
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Unsloth: Offloading embeddings to RAM to save 1.08 GB.


### Reasoning Effort
The `gpt-oss` models from OpenAI include a feature that allows users to adjust the model's "reasoning effort." This gives you control over the trade-off between the model's performance and its response speed (latency) which by the amount of token the model will use to think.

----

The `gpt-oss` models offer three distinct levels of reasoning effort you can choose from:

* **Low**: Optimized for tasks that need very fast responses and don't require complex, multi-step reasoning.
* **Medium**: A balance between performance and speed.
* **High**: Provides the strongest reasoning performance for tasks that require it, though this results in higher latency.

In [ ]:
from transformers import TextStreamer

messages = [
    {"role": "user", "content": "Solve x^5 + 3x^4 - 10 = 3."},
]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    return_tensors = "pt",
    return_dict = True,
    reasoning_effort = "low", # **NEW!** Set reasoning effort to low, medium or high
).to("cuda")

_ = model.generate(**inputs, max_new_tokens = 512, streamer = TextStreamer(tokenizer))

<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.
Knowledge cutoff: 2024-06
Current date: 2025-08-13

Reasoning: low

# Valid channels: analysis, commentary, final. Channel must be included for every message.<|end|><|start|>user<|message|>Solve x^5 + 3x^4 - 10 = 3.<|end|><|start|>assistant<|channel|>analysis<|message|>Equation: x^5 +3x^4 -10 =3 => x^5 +3x^4 -13 =0. Need roots. Could factor numeric? Try integer roots: ±1, ±13. f(1)=1+3-13=-9. f( -1)= -1+3-13=-1? wait: f(-1)=(-1)+3-13=-12. f(13)=371293+3*28561-13 huge positive. f(-13)= -371293+3*28561-13 = -371293+85683-13= -285724. Maybe rational ±13? none. So only real root? Let's find approximate. Use numerical. Let g(x)=x^5+3x^4-13. At x=1: -9. At x=2:32+ 3*16=48 =>80-13=67 positive. So one root between1 and2. Try1.2: 1.2^5= (1.2^2=1.44, *1.2^3? 1.2^3=1.728, *1.2^5 =1.728*1.44= asym? Let's compute: 1.2^4=2.0736, times 1.2 =2.494. So x^5=2.49. 3x^4=3*2.0736=6.2208 sum=8.71-13=-4.29. So still negative

Changing the `reasoning_effort` to `medium` will make the model think longer. We have to increase the `max_new_tokens` to occupy the amount of the generated tokens but it will give better and more correct answer

In [ ]:
from transformers import TextStreamer

messages = [
    {"role": "user", "content": "Solve x^5 + 3x^4 - 10 = 3."},
]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    return_tensors = "pt",
    return_dict = True,
    reasoning_effort = "medium", # **NEW!** Set reasoning effort to low, medium or high
).to("cuda")

_ = model.generate(**inputs, max_new_tokens = 1024, streamer = TextStreamer(tokenizer))

<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.
Knowledge cutoff: 2024-06
Current date: 2025-08-13

Reasoning: medium

# Valid channels: analysis, commentary, final. Channel must be included for every message.<|end|><|start|>user<|message|>Solve x^5 + 3x^4 - 10 = 3.<|end|><|start|>assistant<|channel|>analysis<|message|>We need to solve equation: x^5 + 3x^4 - 10 = 3. So rewrite: x^5 + 3x^4 - 10 = 3 => x^5 + 3x^4 - 13 = 0. So solve polynomial equation x^5 + 3x^4 - 13 = 0. We likely seek real solutions. It's quintic, so analytic roots may be messy. But maybe there's an integer root? We can test integer divisors of 13: ±1,±13. Plug x=1: 1+3-13=-9 not zero. x= -1: -1+3-13=-10. x=13 huge. x = -13: -371293 + 121663 -13 huge negative. So no integer root. Maybe rational? ±13 maybe not. So probably we can factor something? Let’s try substitution x^4(x+3)=13? Actually equation: x^5 + 3x^4 = 13 so x^4(x+3)=13 => x+3 = 13/x^4 => x = 13/x^4 - 3. Hard.

Maybe we f

Lastly we will test it using `reasoning_effort` to `high`

In [ ]:
from transformers import TextStreamer

messages = [
    {"role": "user", "content": "Solve x^5 + 3x^4 - 10 = 3."},
]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    return_tensors = "pt",
    return_dict = True,
    reasoning_effort = "high", # **NEW!** Set reasoning effort to low, medium or high
).to("cuda")

_ = model.generate(**inputs, max_new_tokens = 2048, streamer = TextStreamer(tokenizer))

<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.
Knowledge cutoff: 2024-06
Current date: 2025-08-13

Reasoning: high

# Valid channels: analysis, commentary, final. Channel must be included for every message.<|end|><|start|>user<|message|>Solve x^5 + 3x^4 - 10 = 3.<|end|><|start|>assistant<|channel|>analysis<|message|>The problem: "Solve x^5 + 3x^4 - 10 = 3." It is an equation: x^5 + 3x^4 - 10 = 3. So bring everything: x^5 + 3x^4 - 13 = 0. We need to solve for x? Likely real solutions? Or approximate? Could be factorable? Let's explore algebraically.

Given x^5 + 3x^4 - 13 = 0.

We can attempt to factor by grouping? Not obviously. Maybe try rational root theorem: possible rational roots are factors of 13: ±1, ±13. Let's test:

x=1: 1 + 3 -13 = -9 ≠ 0
x=-1: -1 + 3 -13 = -12 ≠ 0
x=13: huge > 0.
x=-13: negative large. So no integer roots. Maybe something like x=2? Wait rational roots must divide constant 13, but polynomial is degree 5 with leading coeff

And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other resources:
1. Train your own reasoning model - Llama GRPO notebook [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.1_(8B)-GRPO.ipynb)
2. Saving finetunes to Ollama. [Free notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)
3. Llama 3.2 Vision finetuning - Radiography use case. [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.2_(11B)-Vision.ipynb)
4. See notebooks for DPO, ORPO, Continued pretraining, conversational finetuning and more on our [documentation](https://unsloth.ai/docs/get-started/unsloth-notebooks)!

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️
</div>

  This notebook and all Unsloth notebooks are licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).

In [5]:
!pip install -q -U "datasets>=4.8.0" "pyarrow>=20.0.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 16.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unsloth-zoo 2026.9.8 requires datasets!=4.0.*,!=4.1.0,!=4.4.*,!=4.5.0,<5.0.0,>=3.4.1, but you have datasets 5.0.1 which is incompatible.


In [8]:
import json
import requests
from datasets import Dataset

url = (
    "https://huggingface.co/datasets/"
    "MoreThought/Fable-5.1-Max-Reasoning-Filtered-10000x/"
    "resolve/main/5k/train.jsonl?download=true"
)

pairs = []

with requests.get(url, stream=True, timeout=120) as response:
    response.raise_for_status()

    for raw_line in response.iter_lines(decode_unicode=True):
        if not raw_line:
            continue

        row = json.loads(raw_line)
        messages = row.get("messages", [])

        # 从后面找最后一个助手回答
        answer_index = next(
            (
                i for i in range(len(messages) - 1, -1, -1)
                if messages[i].get("role") == "assistant"
                and isinstance(messages[i].get("content"), str)
                and messages[i]["content"].strip()
            ),
            None,
        )

        if answer_index is None:
            continue

        # 找这个回答之前最近的用户问题
        user_index = next(
            (
                i for i in range(answer_index - 1, -1, -1)
                if messages[i].get("role") == "user"
                and isinstance(messages[i].get("content"), str)
                and messages[i]["content"].strip()
            ),
            None,
        )

        if user_index is None:
            continue

        prompt = messages[user_index]["content"].strip()
        completion = messages[answer_index]["content"].strip()

        # 问题最多保留最后 256 tokens；回答最多保留前 512 tokens
        prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"][-256:]
        completion_ids = tokenizer(completion, add_special_tokens=False)["input_ids"][:512]

        if len(completion_ids) < 32:
            continue

        pairs.append({
            "prompt": [{
                "role": "user",
                "content": tokenizer.decode(prompt_ids),
            }],
            "completion": [{
                "role": "assistant",
                "content": tokenizer.decode(completion_ids),
            }],
        })

        if len(pairs) >= 200:
            break

print("可训练样本：", len(pairs))
assert len(pairs) >= 30, "仍然不足 30 条，请把输出贴给我"

data = Dataset.from_list(pairs).train_test_split(
    test_size=0.1,
    seed=42,
)

print(data)
print(data["train"][0])

可训练样本： 200
DatasetDict({
    train: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 180
    })
    test: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 20
    })
})
{'prompt': [{'content': ' ×4\n\nMISSING ITEMS\n=============\nCatalog item 1:\nproject: **py-ontap-volume-job** — Python ONTAP REST volume provisioner using current REST/client library: SVM/aggregate resolution, field selection, POST payload, asynchronous job links, terminal job error handling, and UUID-based idempotent lookup.\n\nCatalog item 2:\nproject: **powershell-ontap-snapshot-pages** — PowerShell ONTAP snapshot inventory/retention client: cluster-management `/api` endpoint, fields/max_records/order_by query handling, `_links.next` pagination, safe DELETE selection, and HAL+JSON errors.\n\nCatalog item 3:\nfeature: **java-ontap-cluster-whoami** — Java ONTAP 9.19.1+ identity/capability probe: call the current whoami endpoint, request explicit fields, fall back cleanly f

In [9]:
from trl import SFTConfig, SFTTrainer

assert "model" in globals(), "找不到 model；请先运行 GPT-OSS 加载格和 LoRA 格"
assert "tokenizer" in globals(), "找不到 tokenizer；请先运行 GPT-OSS 加载格"

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=data["train"],
    eval_dataset=data["test"],
    args=SFTConfig(
        output_dir="/content/fable-training-output",
        max_length=1024,
        completion_only_loss=True,

        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=4,

        max_steps=20,
        learning_rate=2e-5,
        logging_steps=1,

        save_strategy="no",
        report_to="none",
    ),
)

before = trainer.evaluate()
print("训练前测试 loss：", before["eval_loss"])

trainer.train()

after = trainer.evaluate()
print("训练后测试 loss：", after["eval_loss"])

Unsloth: Switching to float32 training since model cannot work with float16


Unsloth: Tokenizing ["prompt"+"completion"] (num_proc=2):   0%|          | 0/180 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["prompt"+"completion"] (num_proc=2):   0%|          | 0/20 [00:00<?, ? examples/s]

ValueError: You cannot perform fine-tuning on purely quantized models. Please attach trainable adapters on top of the quantized model to correctly perform fine-tuning. Please see: https://huggingface.co/docs/transformers/peft for more details